# sanitise - Phase 3a backend on Colab

Runs the full system with the real model on a T4 GPU: Agents 1-3 over the corpus, the
document store and hybrid search index, the HTTP API, and a public address for the dashboard.

`Runtime -> Change runtime type -> T4 GPU`, then run the cells in order.

| Cell | What it does | Time |
|---|---|---|
| ① Setup | Clone `main`, install packages, Ollama and Agent 1's model; Agents 2-3 on Gemini (key in Colab secrets) or a local model | ~5 min |
| ② Evaluate | Agents 1-3 on the 20 fixtures; prints a report to copy | ~3 min |
| ③ Build | Sanitise the corpus into the store and build the search index | ~1 h for 300 docs |
| ③b Save | Save the store and index to Google Drive (tens of MB) | ~1 min |
| ③c Restore | Demo day: bring the store and index back from Drive instead of building | ~1 min |
| ④ Serve | Start the API and a public tunnel (ngrok, or Cloudflare as a fallback); prints the address for the dashboard | ~1 min |

③ resumes where it stopped, so if Colab disconnects, run ① and ③ again.

**Tunnel, once:** make a free account at ngrok.com, copy *Your Authtoken* and claim the free
static domain under *Domains*. In Colab, open Secrets (key icon on the left) and add
`NGROK_AUTHTOKEN` and `NGROK_DOMAIN` with notebook access on. ④ then always prints the same
address, so `web/.env.local` is set once. Without them ④ uses a Cloudflare quick tunnel,
which needs no account but gets a new address each time and can be refused when busy.

**Demo day:** run ① → ③c → ④ → ⑤, about 5-8 minutes (the model and the search model are
downloaded again in every session). Start about 10 minutes early; ⑤ also loads the search
model, so the first question in the demo is fast.

In [ ]:
#@title ① Setup
MODEL = "llama3:8b"  #@param ["llama3:8b", "qwen2.5:7b-instruct"]
#@markdown **Agents 2 and 3.** `gemini`: the Gemini API (free tier). Add your key once as a Colab
#@markdown secret (key icon on the left, notebook access on) named `GEMINI_API_KEY`; several keys,
#@markdown comma-separated, are used in turn when one reaches its daily limit (each from a
#@markdown different Google Cloud project). `ollama`: a local model, AGENTS_MODEL below.
AGENTS_BACKEND = "gemini"  #@param ["gemini", "ollama"]
GEMINI_MODEL = "gemini-3.1-flash-lite"  #@param {type:"string"}
#@markdown For `ollama`: Agents 2-3's model ("" = the same as MODEL) and its thinking switch.
AGENTS_MODEL = ""  #@param ["", "qwen3.5:4b", "qwen3.5:9b"]
AGENTS_THINK = True  #@param {type:"boolean"}
BRANCH = "main"  #@param {type:"string"}

import os
import subprocess
import sys
import time
from pathlib import Path

REPO = Path("/content/3rd_sem_el")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH,
                    "https://github.com/raksha-sendhil/3rd_sem_el.git", str(REPO)], check=True)
os.chdir(REPO)
!git log --oneline -1

!pip install -q -r requirements.txt
!pip install -q -e ".[retrieval]" --no-deps
!pip install -q "sentence-transformers>=3.0" "chromadb>=0.5"
sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(REPO / "tools"))
# Running this cell again after a pull must use the new code: forget modules already loaded.
for name in [m for m in sys.modules if m == "sanitise" or m.startswith(("sanitise.", "colab_cache"))]:
    del sys.modules[name]

import colab_cache

colab_cache.install_ollama_and_model(MODEL)
if AGENTS_BACKEND == "ollama" and AGENTS_MODEL:
    colab_cache.install_ollama_and_model(AGENTS_MODEL)

os.environ.update({
    "SANITISE_LLM_BACKEND": "ollama",
    "SANITISE_MODEL": MODEL,
    "SANITISE_LLM_TIMEOUT_S": "600",
    "SANITISE_RETRIEVAL_MODE": "hybrid",
})
for key in ("SANITISE_AGENTS_BACKEND", "SANITISE_AGENTS_MODEL", "SANITISE_AGENTS_THINK",
            "SANITISE_GEMINI_API_KEY"):
    os.environ.pop(key, None)
if AGENTS_BACKEND == "gemini":
    from google.colab import userdata
    try:
        gemini_key = userdata.get("GEMINI_API_KEY")
    except Exception as exc:
        raise RuntimeError("Add the Colab secret GEMINI_API_KEY (key icon on the left) and give "
                           "this notebook access, then run this cell again.") from exc
    os.environ.update({"SANITISE_AGENTS_BACKEND": "gemini", "SANITISE_AGENTS_MODEL": GEMINI_MODEL,
                       "SANITISE_GEMINI_API_KEY": gemini_key})
    AGENTS = f"{GEMINI_MODEL} (Gemini API, {len(gemini_key.split(','))} key(s))"
elif AGENTS_MODEL:
    os.environ["SANITISE_AGENTS_MODEL"] = AGENTS_MODEL
    os.environ["SANITISE_AGENTS_THINK"] = "true" if AGENTS_THINK else "false"
    AGENTS = f"{AGENTS_MODEL} (thinking {'on' if AGENTS_THINK else 'off'})"
else:
    AGENTS = MODEL
print("ready: Agent 1", MODEL, "| Agents 2-3", AGENTS)

In [ ]:
#@title ② Evaluate Agents 1-3 on the 20 fixtures (copy the report at the end)

from sanitise.agents.pipeline import sanitise
from sanitise.config import get_settings
from sanitise.errors import PipelineError
from sanitise.llm.ollama import OllamaClient
from sanitise.loader import iter_documents

get_settings.cache_clear()
records, failures = [], {}
async with OllamaClient.from_settings(get_settings()) as llm:
    for doc in iter_documents("data/fixtures"):
        try:
            records.append(await sanitise(doc.doc_id, doc.text, llm))
        except PipelineError as exc:
            failures[doc.doc_id] = str(exc)
        print(".", end="", flush=True)
print()

out = Path("/content/eval_fixtures")
out.mkdir(exist_ok=True)
for r in records:
    (out / f"{r.doc_id}.json").write_text(r.model_dump_json(indent=2))

lines = [f"AGENT 1 {MODEL}  AGENTS 2-3 {AGENTS}  documents {len(records)}  failures {len(failures)}"]
lines.append(f"leak retries {sum(r.meta.leak_retries for r in records)}  "
             f"residual leaks {sum(len(r.residual_leaks) for r in records)}  "
             f"schema retries {sum(r.meta.retries for r in records)}  "
             f"masked words {sum(r.rewriter.safe_text.count('[REDACTED]') for r in records)}  "
             f"avg seconds {sum(r.meta.total_ms for r in records) / max(len(records), 1) / 1000:.1f}")
if Path("data/gold_facts").exists() and any(Path("data/gold_facts").glob("*.json")):
    from sanitise.evaluation.facts import score_facts
    from sanitise.schemas import ExtractorOutput, GoldRouting
    gold_facts = [ExtractorOutput.model_validate_json(p.read_text())
                  for p in sorted(Path("data/gold_facts").glob("*.json"))]
    lines.append(f"facts: {score_facts([r.extractor for r in records], gold_facts)}")
    if Path("data/gold_rewrite").exists() and any(Path("data/gold_rewrite").glob("*.json")):
        from sanitise.evaluation.rewrite import score_rewrite
        gold_routing = [GoldRouting.model_validate_json(p.read_text())
                        for p in sorted(Path("data/gold_rewrite").glob("*.json"))]
        lines.append(f"rewrite: {score_rewrite([r.rewriter for r in records], gold_routing, gold_facts)}")
for doc_id, err in failures.items():
    lines.append(f"FAILED {doc_id}: {err[:200]}")
for r in records:
    lines.append(f"--- {r.doc_id}  facts={len(r.extractor.facts)}  leak_retries={r.meta.leak_retries}")
    lines.append("FACTS: " + "; ".join(f"{f.subject}.{f.relation}={f.value}" for f in r.extractor.facts[:20]))
    lines.append("SAFE: " + r.rewriter.safe_text.replace(chr(10), " "))
report = "\n".join(lines)
Path("/content/eval_report.txt").write_text(report)
print(report)

In [ ]:
#@title ③ Build the store and search index from the corpus (resumes after a disconnect)
CORPUS = "data/corpus/docs" if any(Path("data/corpus/docs").glob("*.json")) else "data/fixtures"
print("corpus:", CORPUS)
!sanitise build --input {CORPUS} --store data/store.sqlite --out data/processed

In [ ]:
#@title ③b Save the store and index to Google Drive
from google.colab import drive

drive.mount("/content/drive")
SAVED = Path("/content/drive/MyDrive/sanitise/store.tar.gz")
SAVED.parent.mkdir(parents=True, exist_ok=True)
!tar -czf /content/store.tar.gz data/store.sqlite data/index
!cp /content/store.tar.gz "{SAVED}"
print(f"saved {SAVED.stat().st_size / 1e6:.1f} MB to {SAVED}")

In [ ]:
#@title ③c Demo day: restore the store and index from Google Drive (instead of ③)
from google.colab import drive

drive.mount("/content/drive")
SAVED = Path("/content/drive/MyDrive/sanitise/store.tar.gz")
!tar -xzf "{SAVED}" -C "{REPO}"
print(f"restored data/store.sqlite ({Path('data/store.sqlite').stat().st_size / 1e6:.1f} MB) and data/index")

In [ ]:
#@title ④ Start the API and a public tunnel for the dashboard
#@markdown **Tunnel.** ngrok when its two Colab secrets are set (key icon on the left, notebook
#@markdown access on): `NGROK_AUTHTOKEN` (ngrok.com, free account -> Your Authtoken) and
#@markdown `NGROK_DOMAIN` (Domains -> your free static domain, e.g. `name.ngrok-free.app`).
#@markdown The address then never changes. Otherwise, or if ngrok fails, a Cloudflare quick
#@markdown tunnel (no account, a new address each time, no uptime guarantee).
import re
import sys
import urllib.request


def api_health():
    try:
        return urllib.request.urlopen("http://127.0.0.1:8000/v1/health", timeout=2).read().decode()
    except Exception:
        return None


colab_cache.start_ollama()
if api_health() is None:  # running this cell again keeps the API that is already up
    api_log = open("/content/api.log", "a")
    api = subprocess.Popen(["sanitise", "serve", "--host", "0.0.0.0", "--port", "8000"],
                           stdout=api_log, stderr=subprocess.STDOUT, env=os.environ.copy())
    for _ in range(120):
        if api_health():
            break
        time.sleep(1)
print(api_health() or "API did not start: see /content/api.log")


def colab_secret(name):
    try:
        from google.colab import userdata
        return (userdata.get(name) or "").strip() or None
    except Exception:
        return None


def ngrok_tunnel():
    token = colab_secret("NGROK_AUTHTOKEN")
    if not token:
        print("ngrok: no NGROK_AUTHTOKEN secret, skipped")
        return None
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok"], check=True)
    from pyngrok import conf, ngrok
    conf.get_default().auth_token = token
    ngrok.kill()
    domain = colab_secret("NGROK_DOMAIN")
    domain = re.sub(r"^https?://", "", domain).rstrip("/") if domain else None
    # newer ngrok agents take the fixed address as "url", older ones as "domain"
    attempts = [{"url": "https://" + domain}, {"domain": domain}] if domain else [{}]
    for options in attempts:
        try:
            return ngrok.connect(8000, "http", **options).public_url
        except Exception as exc:
            print("ngrok failed:", str(exc)[:300])
    return None


def cloudflare_tunnel(attempts=5):
    if not Path("/usr/local/bin/cloudflared").exists():
        subprocess.run(["wget", "-q", "-O", "/usr/local/bin/cloudflared",
                        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
        subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)
    subprocess.run(["pkill", "-f", "cloudflared"])
    for attempt in range(1, attempts + 1):
        log = open("/content/tunnel.log", "w")
        tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
                                  stdout=log, stderr=subprocess.STDOUT)
        for _ in range(90):
            text = Path("/content/tunnel.log").read_text()
            found = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
            if found:
                return found[0]
            if "429" in text or tunnel.poll() is not None:
                break
            time.sleep(1)
        tunnel.kill()
        if attempt < attempts:
            print(f"cloudflare: attempt {attempt} failed, retrying in {30 * attempt} s")
            time.sleep(30 * attempt)
    print("cloudflare failed:", Path("/content/tunnel.log").read_text()[-800:])
    return None


URL = ngrok_tunnel() or cloudflare_tunnel()
if URL:
    print("API address for the dashboard:", URL)
    print("Set it in web/.env.local as NEXT_PUBLIC_API_URL=" + URL)
else:
    print("No tunnel. The API is still running; run this cell again in a few minutes.")

In [ ]:
#@title ⑤ Smoke test through the tunnel
import json as _json

HEADERS = {"content-type": "application/json", "ngrok-skip-browser-warning": "1"}

req = urllib.request.Request(
    URL + "/v1/sessions", data=b"{}", headers=HEADERS, method="POST")
session = _json.load(urllib.request.urlopen(req, timeout=60))["session_id"]
body = _json.dumps({"query": "Which patient has vertigo?", "session_id": session}).encode()
req = urllib.request.Request(URL + "/v1/compare", data=body,
                             headers=HEADERS, method="POST")
result = _json.load(urllib.request.urlopen(req, timeout=600))
print("BASELINE: ", result["baseline"]["answer"])
print("PROTECTED:", result["protected"]["answer"])